# Feature Selection CV Comparison — CICIoMT2024 (Google Colab)

Compares four feature-selection methods **without using the official test set** for method choice.

| Item | Detail |
|------|--------|
| **Platform** | Google Colab — **Runtime → Run all** |
| **GPU** | **Runtime → Change runtime type → T4 GPU** (recommended) |
| **Dataset** | Train CSV auto-download from Kaggle via Colab Secrets |
| **Methods** | Correlation · Mutual Information · RFE · XGB Gain |
| **Protocol** | Stratified K-Fold on train only; SMOTE on train fold; score validation fold |
| **Output** | `/content/outputs/fs_cv_comparison.csv` and plots |

> Cite this notebook as empirical justification for choosing XGB Gain in the main pipeline.


---
## Cell 1 — Setup


In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU detected (XGBoost will use CUDA when supported):')
    for line in result.stdout.split('\n'):
        if any(k in line for k in ('NVIDIA', 'Tesla', 'GeForce')):
            print(' ', line.strip())
else:
    print('No GPU — enable Runtime > Change runtime type > T4 GPU.')

!pip install -q --upgrade xgboost scikit-learn imbalanced-learn psutil joblib kaggle kagglehub

import xgboost as xgb
import sklearn
print(f'\nXGBoost : {xgb.__version__}')
print(f'Sklearn : {sklearn.__version__}')
print('Packages ready.')


---
## Cell 2 — Kaggle Auth & Dataset Download

1. Open Colab **Secrets** (sidebar key icon).
2. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` from [kaggle.com/settings](https://www.kaggle.com/settings).
3. Enable **Notebook access** for both secrets.
4. Re-run this cell if download fails after adding secrets.


In [ ]:
import json, os, subprocess
from pathlib import Path

KAGGLE_DATASET = 'limamateus/cic-iomt-2024-wifi-mqtt'
DATA_DIR  = Path('/content/ciciomt2024')
MODEL_DIR = Path('/content/models')
OUTPUT_DIR = Path('/content/outputs')
for p in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_train.csv'
TEST_FILE  = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_test.csv'


def setup_kaggle_credentials():
    kaggle_dir = Path('/root/.kaggle')
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        print('Using existing /root/.kaggle/kaggle.json')
        return

    username, key = None, None
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        print('Kaggle credentials loaded from Colab Secrets.')
    except Exception:
        username = os.environ.get('KAGGLE_USERNAME')
        key = os.environ.get('KAGGLE_KEY')
        if username and key:
            print('Kaggle credentials loaded from environment variables.')

    if not username or not key:
        raise RuntimeError(
            'Kaggle credentials not found.\n'
            'Colab sidebar > Secrets > add KAGGLE_USERNAME and KAGGLE_KEY, '
            'then enable Notebook access.'
        )

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    with open(kaggle_json, 'w', encoding='utf-8') as f:
        json.dump({'username': username, 'key': key}, f)
    os.chmod(kaggle_json, 0o600)
    print('kaggle.json created.')


def ensure_dataset(require_test=True):
    global TRAIN_FILE, TEST_FILE
    if TRAIN_FILE.exists() and (not require_test or TEST_FILE.exists()):
        print(f'Dataset already present in {DATA_DIR}')
        return

    setup_kaggle_credentials()
    print(f'Downloading {KAGGLE_DATASET} ...')
    try:
        import kagglehub
        dl_path = kagglehub.dataset_download(KAGGLE_DATASET)
        print(f'kagglehub path: {dl_path}')
        for csv in Path(dl_path).rglob('*.csv'):
            target = DATA_DIR / csv.name
            if not target.exists():
                target.write_bytes(csv.read_bytes())
    except Exception as e1:
        print(f'kagglehub failed ({e1}), trying kaggle CLI...')
        subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
             '-p', str(DATA_DIR), '--unzip', '-q'],
            check=False,
        )

    if not TRAIN_FILE.exists():
        found = list(DATA_DIR.rglob('*train*.csv'))
        if found:
            TRAIN_FILE = found[0]
        else:
            raise FileNotFoundError(f'Train CSV not found under {DATA_DIR}')
    if require_test and not TEST_FILE.exists():
        found = list(DATA_DIR.rglob('*test*.csv'))
        if found:
            TEST_FILE = found[0]
        else:
            raise FileNotFoundError(f'Test CSV not found under {DATA_DIR}')

    print(f'\nTrain: {TRAIN_FILE}')
    if require_test:
        print(f'Test : {TEST_FILE}')


ensure_dataset(require_test=False)


---
## Cell 3 — Imports & Settings


In [ ]:
import os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, roc_auc_score,
    precision_recall_fscore_support,
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from IPython.display import display

# Colab paths (Cell 2)
DATA_DIR  = Path('/content/ciciomt2024')
OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_train.csv'

RANDOM_SEED = 42
MAX_ROWS = int(os.getenv('MAX_ROWS', '500000'))
CV_MAX_ROWS = int(os.getenv('CV_MAX_ROWS', '150000'))
CV_FOLDS = int(os.getenv('CV_FOLDS', '3'))
TOP_N = 20
FS_METHODS = ['Correlation', 'Mutual Information', 'RFE', 'XGB Gain']
np.random.seed(RANDOM_SEED)

try:
    XGBClassifier(device='cuda', n_estimators=1)
    XGB_DEVICE = 'cuda'
except Exception:
    XGB_DEVICE = 'cpu'

print(f'XGB device : {XGB_DEVICE}')
print(f'Data dir   : {DATA_DIR}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'CV folds   : {CV_FOLDS} | CV max rows: {CV_MAX_ROWS:,}')


In [ ]:
DROP_COLS = ['label', 'binary_label', 'flow_id', 'Flow ID',
             'src_ip', 'Src IP', 'dst_ip', 'Dst IP']

def load_with_label(filepath):
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    existing = [c for c in df.columns if c.lower() in ('label', 'class', 'attack', 'type')]
    if existing:
        df.rename(columns={existing[0]: 'label'}, inplace=True)
    df['label'] = df['label'].astype(str).str.strip()
    return df

def to_attack_binary(label):
    text = str(label).lower()
    return 1 if ('ddos' in text or 'dos' in text) else 0

def get_Xy(df, target='binary_label'):
    drop = [c for c in DROP_COLS if c in df.columns]
    X = df.drop(columns=drop).copy()
    y = df[target].copy().reset_index(drop=True)
    return X, y

print('Loading train CSV...')
df_train = load_with_label(TRAIN_FILE)
df_train['binary_label'] = df_train['label'].apply(to_attack_binary)

X_raw, y = get_Xy(df_train)
label_encoders = {}
for col in X_raw.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))
    label_encoders[col] = le

X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
miss_pct = X_raw.isnull().mean()
drop_miss = miss_pct[miss_pct > 0.5].index.tolist()
if drop_miss:
    X_raw.drop(columns=drop_miss, inplace=True)
train_medians = X_raw.median(numeric_only=True)
X_raw.fillna(train_medians, inplace=True)
X_raw = X_raw.select_dtypes(include=[np.number]).reset_index(drop=True)
feature_columns = X_raw.columns.tolist()

mask = ~X_raw.duplicated()
X_raw, y = X_raw[mask].reset_index(drop=True), y[mask].reset_index(drop=True)

if len(X_raw) > MAX_ROWS:
    rng = np.random.default_rng(RANDOM_SEED)
    parts = []
    for cls in y.unique():
        idx = y[y == cls].index.to_numpy()
        n_take = min(int(MAX_ROWS * len(idx) / len(y)), len(idx))
        parts.append(rng.choice(idx, n_take, replace=False))
    sampled_idx = np.concatenate(parts)
    rng.shuffle(sampled_idx)
    X_raw = X_raw.iloc[sampled_idx].reset_index(drop=True)
    y = y.iloc[sampled_idx].reset_index(drop=True)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)
y_train = y.reset_index(drop=True)

print(f'Train matrix: {X_train.shape}')
print(f'Class 0: {(y_train==0).sum():,} | Class 1: {(y_train==1).sum():,}')
print('Official test set is NOT used in this notebook.')


---
## Cell 3 — FS Methods & CV Helpers


In [ ]:
def _resample_fold(X_tr, y_tr, random_state=RANDOM_SEED):
    """SMOTE or undersample train fold to ~10:1 (Class0:Class1) when imbalanced."""
    n0, n1 = int((y_tr == 0).sum()), int((y_tr == 1).sum())
    target_ratio = 10.0
    current = n0 / max(n1, 1)
    n0_target = max(1, int(round(n1 * target_ratio)))
    n1_target = max(1, int(round(n0 / target_ratio)))
    if abs(current - target_ratio) / target_ratio < 1e-6:
        return X_tr, y_tr
    if current > target_ratio and n0 > n0_target:
        rus = RandomUnderSampler(sampling_strategy={0: n0_target}, random_state=random_state)
        X_r, y_r = rus.fit_resample(X_tr, y_tr)
    elif current < target_ratio and n0 < n0_target:
        k = min(5, n0 - 1)
        if k < 1:
            return X_tr, y_tr
        sm = SMOTE(sampling_strategy={0: n0_target}, random_state=random_state, k_neighbors=k)
        X_r, y_r = sm.fit_resample(X_tr, y_tr)
    else:
        return X_tr, y_tr
    return pd.DataFrame(X_r, columns=X_tr.columns), pd.Series(y_r, name='binary_label')


def select_correlation(X_fit, y_fit, top_n=TOP_N):
    corr_mat = X_fit.corr().abs()
    upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    drop_redun = [c for c in upper.columns if any(upper[c] > 0.90)]
    X_f = X_fit.drop(columns=drop_redun, errors='ignore')
    y_s = pd.Series(y_fit.values, index=X_f.index)
    target_corr = X_f.corrwith(y_s).abs().sort_values(ascending=False)
    feats = target_corr.head(top_n).index.tolist()
    return feats


def select_mutual_info(X_fit, y_fit, top_n=TOP_N):
    scores = mutual_info_classif(X_fit, y_fit, random_state=RANDOM_SEED)
    mi_df = pd.DataFrame({'feature': X_fit.columns, 'score': scores})
    return mi_df.sort_values('score', ascending=False).head(top_n)['feature'].tolist()


def select_rfe(X_fit, y_fit, top_n=TOP_N):
    est = DecisionTreeClassifier(max_depth=8, random_state=RANDOM_SEED)
    rfe = RFE(estimator=est, n_features_to_select=top_n, step=5)
    rfe.fit(X_fit, y_fit)
    return X_fit.columns[rfe.support_].tolist()


def select_xgb_gain(X_fit, y_fit, top_n=TOP_N):
    n0, n1 = int((y_fit == 0).sum()), int((y_fit == 1).sum())
    spw = n0 / max(n1, 1)
    model = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.2,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, importance_type='gain',
        device=XGB_DEVICE, random_state=RANDOM_SEED,
        verbosity=0, eval_metric='logloss')
    model.fit(X_fit, y_fit)
    imp = pd.DataFrame({'feature': X_fit.columns, 'importance': model.feature_importances_})
    return imp.sort_values('importance', ascending=False).head(top_n)['feature'].tolist()


FS_SELECTORS = {
    'Correlation': select_correlation,
    'Mutual Information': select_mutual_info,
    'RFE': select_rfe,
    'XGB Gain': select_xgb_gain,
}


def _fold_metrics(y_true, y_pred, y_prob):
    rec = recall_score(y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    return {
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Macro_F1': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'Class_0_Recall': round(float(rec[0]), 4),
        'Class_1_Recall': round(float(rec[1]), 4),
        'AUC': round(roc_auc_score(y_true, y_prob), 4) if len(np.unique(y_true)) > 1 else None,
    }


def _subset_cv_data(X, y, max_rows):
    if len(X) <= max_rows:
        return X.reset_index(drop=True), y.reset_index(drop=True)
    idx, _ = train_test_split(
        np.arange(len(y)), train_size=max_rows, random_state=RANDOM_SEED, stratify=y)
    return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)


---
## Cell 4 — Run Stratified K-Fold CV (Train Only)


In [ ]:
X_cv, y_cv = _subset_cv_data(X_train, y_train, CV_MAX_ROWS)
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_rows = []
fold_features = {m: [] for m in FS_METHODS}

print(f'CV subset: {len(X_cv):,} rows | {CV_FOLDS} folds')
print('Official test set is NOT used.\n')

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_cv, y_cv)):
    X_tr = X_cv.iloc[tr_idx]
    y_tr = y_cv.iloc[tr_idx]
    X_va = X_cv.iloc[va_idx]
    y_va = y_cv.iloc[va_idx]

    X_fit, y_fit = _resample_fold(X_tr, y_tr)
    n0, n1 = int((y_fit == 0).sum()), int((y_fit == 1).sum())
    spw = n0 / max(n1, 1)

    for method_name, selector_fn in FS_SELECTORS.items():
        t0 = time.time()
        try:
            features = selector_fn(X_fit, y_fit)
        except Exception as exc:
            print(f'  Fold {fold_idx+1} {method_name}: FAILED ({exc})')
            continue
        fs_time = time.time() - t0
        fold_features[method_name].append(set(features))

        X_fit_sel = X_fit[features]
        X_va_sel = X_va[features]

        model = XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw, device=XGB_DEVICE,
            random_state=RANDOM_SEED, verbosity=0, eval_metric='logloss')
        t1 = time.time()
        model.fit(X_fit_sel, y_fit)
        train_time = time.time() - t1

        y_pred = model.predict(X_va_sel)
        y_prob = model.predict_proba(X_va_sel)[:, 1]
        metrics = _fold_metrics(y_va, y_pred, y_prob)

        cv_rows.append({
            'Fold': fold_idx + 1,
            'FS_Method': method_name,
            'Val_Rows': len(va_idx),
            'Train_Rows_After_Resample': len(X_fit),
            'n_features': len(features),
            'FS_Time_s': round(fs_time, 3),
            'Train_Time_s': round(train_time, 3),
            **metrics,
        })
        print(
            f"  Fold {fold_idx+1} | {method_name:<22} "
            f"AUC={metrics['AUC']} Macro-F1={metrics['Macro_F1']} "
            f"Rec0={metrics['Class_0_Recall']} Rec1={metrics['Class_1_Recall']}"
        )

cv_df = pd.DataFrame(cv_rows)
cv_summary = (
    cv_df.groupby('FS_Method', as_index=False)
    .agg(
        Folds=('Fold', 'count'),
        AUC_mean=('AUC', 'mean'),
        AUC_std=('AUC', 'std'),
        Macro_F1_mean=('Macro_F1', 'mean'),
        Macro_F1_std=('Macro_F1', 'std'),
        Class_0_Recall_mean=('Class_0_Recall', 'mean'),
        Class_1_Recall_mean=('Class_1_Recall', 'mean'),
        Accuracy_mean=('Accuracy', 'mean'),
        FS_Time_mean_s=('FS_Time_s', 'mean'),
        Train_Time_mean_s=('Train_Time_s', 'mean'),
    )
    .round(4)
    .sort_values('Macro_F1_mean', ascending=False)
)

print('\n=== FS CV SUMMARY (train-only, mean ± std) ===')
display(cv_summary)
cv_df.to_csv(OUTPUT_DIR / 'fs_cv_comparison_folds.csv', index=False)
cv_summary.to_csv(OUTPUT_DIR / 'fs_cv_comparison.csv', index=False)
print(f'Saved: {OUTPUT_DIR / "fs_cv_comparison.csv"}')


---
## Cell 5 — Feature Overlap & Visualisation


In [ ]:
# Jaccard overlap of top-20 features across methods (per fold, then average)
overlap_rows = []
methods = FS_METHODS
for i, m1 in enumerate(methods):
    for m2 in methods[i:]:
        scores = []
        for f1, f2 in zip(fold_features[m1], fold_features[m2]):
            if f1 and f2:
                scores.append(len(f1 & f2) / max(len(f1 | f2), 1))
        overlap_rows.append({
            'Method_A': m1,
            'Method_B': m2,
            'Jaccard_mean': round(float(np.mean(scores)), 4) if scores else None,
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(OUTPUT_DIR / 'fs_cv_feature_overlap.csv', index=False)

# Bar chart: Macro-F1 and AUC by FS method
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_df = cv_summary.set_index('FS_Method')
x = np.arange(len(plot_df))
axes[0].bar(x, plot_df['Macro_F1_mean'], yerr=plot_df['Macro_F1_std'], capsize=4, color='steelblue')
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df.index, rotation=15, ha='right')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Macro-F1 (CV validation)')

axes[1].bar(x, plot_df['AUC_mean'], yerr=plot_df['AUC_std'], capsize=4, color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df.index, rotation=15, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('AUC (CV validation)')

# Jaccard heatmap (unique pairs)
mat = pd.DataFrame(1.0, index=methods, columns=methods)
for _, row in overlap_df.iterrows():
    mat.loc[row['Method_A'], row['Method_B']] = row['Jaccard_mean']
    mat.loc[row['Method_B'], row['Method_A']] = row['Jaccard_mean']
sns.heatmap(mat.astype(float), annot=True, fmt='.2f', cmap='Blues', ax=axes[2], vmin=0, vmax=1)
axes[2].set_title('Feature-set overlap (Jaccard)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fs_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR / "fs_cv_comparison.png"}')


---
## Cell 6 — How to Cite in Your Report

**Suggested wording:**

> We compared four feature-selection methods (correlation filtering, mutual information, recursive feature elimination, and XGBoost gain importance) using **stratified K-fold cross-validation on the training set only** (`K=3`, up to 150k stratified samples). For each fold, resampling was applied to the training split only; feature selection was fit on the resampled fold; an XGBoost classifier was trained on the selected top-20 features and evaluated on the held-out validation fold. The official CICIoMT2024 test set was **not** used for method selection. [Report Macro-F1/AUC from `fs_cv_comparison.csv`.] XGB Gain Importance was selected for the final pipeline because it achieved [competitive/best] validation performance while aligning with the final gradient-boosted tree classifier and capturing non-linear feature contributions.

**Files to attach:** `outputs/fs_cv_comparison.csv`, `outputs/fs_cv_feature_overlap.csv`, `outputs/fs_cv_comparison.png`


---
## Download Results (Colab)

Run after experiments finish to save CSVs and plots to your computer.


In [ ]:
from google.colab import files
import shutil
from pathlib import Path

zip_base = '/content/ddos_outputs'
if Path(OUTPUT_DIR).exists() and any(Path(OUTPUT_DIR).iterdir()):
    shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
    print('Downloading outputs zip...')
    files.download(f'{zip_base}.zip')
else:
    print('No files in outputs yet — run experiment cells first.')
